# Edge Computing — Wokwi (ESP32) → Broker MQTT → Python (Paho MQTT)

**Turma:** 1ESPI

| Integrante | RM |
| --- | --- |
| Bruno Carreiro Dos Santos | 569423 |
| Eduardo Bechara Medeiros Craveiro | 571081 |
| Gustavo Moita de Lima | 569180 |
| Daniel Graciano dos Santos Ferreira | 568886 |

**Broker:** `54.91.80.136:1883`

Ordem de execução das células:
1. Instalar o `paho-mqtt`
2. Configuração (`DEVICE_ID = device018`, computador N18)
3. **Subscriber** — recebe a telemetria enviada pelo ESP32
4. **Publisher** — envia os comandos de ligar/desligar o LED

> **Importante:** sempre interrompa a execução anterior antes de rodar novamente,
> evitando múltiplas threads conectadas ao broker.

In [ ]:
!pip install paho-mqtt

## 1. Configuração

O `DEVICE_ID` é `device018` (computador N18) — o mesmo valor usado no código do Wokwi.

In [ ]:
BROKER = "54.91.80.136"
PORT = 1883
KEEPALIVE = 60
DEVICE_ID = "device018"  # computador N18 do laboratorio

TOPICO_TELEMETRIA = f"/TEF/{DEVICE_ID}/attrs"  # ESP32 -> Python
TOPICO_COMANDO = f"/TEF/{DEVICE_ID}/cmd"       # Python -> ESP32

print("Device:", DEVICE_ID)
print("Telemetria:", TOPICO_TELEMETRIA)
print("Comando...:", TOPICO_COMANDO)


## 2. Subscriber — recebendo os dados do ESP32

Rode esta célula com o simulador do Wokwi **em execução**. Para parar, interrompa a célula (botão ■).

In [ ]:
from datetime import datetime

import paho.mqtt.client as mqtt


def criar_cliente(client_id):
    try:  # paho-mqtt >= 2.0
        return mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=client_id)
    except AttributeError:  # paho-mqtt 1.x
        return mqtt.Client(client_id=client_id)


def parse_telemetria(payload):
    """Converte "t|24.0|h|40.0" em {"t": "24.0", "h": "40.0"}."""
    partes = payload.split("|")
    return dict(zip(partes[0::2], partes[1::2]))


def on_connect(client, userdata, flags, reason_code, properties=None):
    print(f"[SUCESSO] Conectado ao Broker. RC: {reason_code}")
    client.subscribe(TOPICO_TELEMETRIA)
    print(f"[SUCESSO] Inscrito em {TOPICO_TELEMETRIA}. Aguardando leituras...")
    print("-" * 60)


def on_message(client, userdata, msg):
    payload = msg.payload.decode("utf-8", errors="replace").strip()
    horario = datetime.now().strftime("%H:%M:%S")
    leitura = parse_telemetria(payload)

    if "t" in leitura and "h" in leitura:
        print(f"[NOVA LEITURA {horario}] Temperatura: {leitura['t']} C | Umidade: {leitura['h']} %")
    else:
        print(f"[NOVA LEITURA {horario}] {payload}")


subscriber = criar_cliente(f"colab_subscriber_{DEVICE_ID}")
subscriber.on_connect = on_connect
subscriber.on_message = on_message
subscriber.connect(BROKER, PORT, keepalive=KEEPALIVE)

try:
    subscriber.loop_forever()
except KeyboardInterrupt:
    print("\n[FIM] Leitura interrompida pelo usuario.")
finally:
    subscriber.disconnect()


## 3. Publisher — enviando comandos para o ESP32

Envia `device018@on|` e `device018@off|` para o tópico de comando.
Acompanhe o LED acendendo/apagando no Wokwi.

In [ ]:
import time

publisher = criar_cliente(f"colab_publisher_{DEVICE_ID}")
publisher.connect(BROKER, PORT, keepalive=KEEPALIVE)
publisher.loop_start()
print(f"[SUCESSO] Conectado. Publicando em {TOPICO_COMANDO}")

try:
    for ciclo in range(1, 6):
        print(f"\nCiclo {ciclo}")
        publisher.publish(TOPICO_COMANDO, f"{DEVICE_ID}@on|")
        print(f"  -> {DEVICE_ID}@on|  (LED ligado)")
        time.sleep(5)
        publisher.publish(TOPICO_COMANDO, f"{DEVICE_ID}@off|")
        print(f"  -> {DEVICE_ID}@off| (LED desligado)")
        time.sleep(5)
except KeyboardInterrupt:
    print("\n[FIM] Publisher interrompido pelo usuario.")
finally:
    publisher.loop_stop()
    publisher.disconnect()
    print("[INFO] Desconectado do broker.")
